# 🔍 Notebook 01b — Exogenous Feature Analysis
**Goal:** Deeply analyse every exogenous feature before using it in any model. This notebook answers:
- Which features actually correlate with transaction volume/value?
- How much does each holiday type suppress or spike volume?
- Which features add predictive power beyond the time series itself?
- Do features interact with each other?
- Which ones should be included vs dropped in SARIMAX / Prophet / LSTM?

## Features Analysed

| Feature | Type | Description |
|---|---|---|
| `Is_Weekend` | Binary | Saturday or Sunday |
| `Is_Weekday` | Binary | Monday–Friday |
| `Is_Festival` | Binary | Indian public holiday |
| `Is_Holiday_Adjacent` | Binary | 1 day before/after holiday |
| `Is_Long_Weekend` | Binary | Weekend touching a holiday |
| `Holiday_Cluster_7D` | Count | Holidays within ±3 days |
| `Days_To_Next_Holiday` | Numeric | Days until next holiday |
| `Days_Since_Prev_Holiday` | Numeric | Days since last holiday |
| `Day_Number` | Ordinal | 0=Mon … 6=Sun |
| `Month_Num` | Ordinal | 1–12 |
| `Quarter` | Ordinal | 1–4 |

## Contents
1. Setup & Data
2. Feature Distributions
3. Univariate Correlation with Target
4. Point-Biserial & ANOVA Tests (statistical significance)
5. Volume by Each Binary Feature (box plots + effect size)
6. Day-of-Week Deep Dive
7. Month & Quarter Seasonality
8. Holiday Cluster & Proximity Effects
9. Feature-Feature Interaction Heatmap
10. Mutual Information & Permutation Importance
11. SARIMAX: With vs Without Exog (model impact)
12. Prophet: Regressor Importance
13. Final Feature Recommendation Table


## 1. Setup & Data

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")
os.makedirs("plots", exist_ok=True)

DATA_PATH  = "data/UPI_Master_2021_2025.csv"
TRAIN_END  = "2025-09-30"
TEST_START = "2025-10-01"

BLUE,RED,ORANGE,GREEN,PURPLE,BROWN,CYAN,PINK = (
    "#1A6FBF","#D62728","#E07B39","#2CA02C","#9467BD","#8C564B","#17BECF","#E377C2")

plt.rcParams.update({
    "figure.facecolor":"#FAFAFA","axes.facecolor":"#FAFAFA",
    "axes.grid":True,"grid.alpha":0.3,"font.family":"DejaVu Sans",
    "axes.spines.top":False,"axes.spines.right":False,
})

df    = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
df["Festival_Name"] = df["Festival_Name"].fillna("")
df["Holiday_Type"]  = df["Holiday_Type"].fillna("Unknown")
df["Month_Num"]     = df["Date"].dt.month
df["Quarter"]       = df["Date"].dt.quarter
df["YearStr"]       = df["Date"].dt.year.astype(str)

train = df[df["Date"] <= TRAIN_END].copy()
test  = df[df["Date"] >= TEST_START].copy()

BINARY_FEATS  = ["Is_Weekend","Is_Festival","Is_Holiday_Adjacent","Is_Long_Weekend"]
COUNT_FEATS   = ["Holiday_Cluster_7D"]
NUMERIC_FEATS = ["Days_To_Next_Holiday","Days_Since_Prev_Holiday"]
ORDINAL_FEATS = ["Day_Number","Month_Num","Quarter"]
ALL_EXOG      = BINARY_FEATS + COUNT_FEATS + NUMERIC_FEATS + ORDINAL_FEATS
TARGETS       = ["Volume (In Mn.)", "Value (In Cr.)"]

print(f"Dataset: {df.shape}  |  Train: {len(train):,}  Test: {len(test):,}")
print(f"\nFeature groups:")
print(f"  Binary  : {BINARY_FEATS}")
print(f"  Count   : {COUNT_FEATS}")
print(f"  Numeric : {NUMERIC_FEATS}")
print(f"  Ordinal : {ORDINAL_FEATS}")
df[ALL_EXOG + TARGETS].describe().round(2)


## 2. Feature Distributions
> Understand the class balance of binary features and distributions of numeric ones before any modelling.

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)

# Binary feature class balance
for i, feat in enumerate(BINARY_FEATS):
    ax = fig.add_subplot(gs[0, i])
    counts = df[feat].value_counts()
    bars = ax.bar(["No (0)","Yes (1)"], [counts.get(0,0), counts.get(1,0)],
                  color=[BLUE, RED], edgecolor="white", width=0.5)
    for bar, val in zip(bars, [counts.get(0,0), counts.get(1,0)]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                f"{val}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=8)
    ax.set_title(feat, fontweight="bold", fontsize=9)
    ax.set_ylabel("Days")

# Count feature
ax = fig.add_subplot(gs[1, 0])
sns.countplot(x="Holiday_Cluster_7D", data=df, palette="Blues", ax=ax)
ax.set_title("Holiday_Cluster_7D", fontweight="bold", fontsize=9)

# Numeric features
for i, feat in enumerate(NUMERIC_FEATS):
    ax = fig.add_subplot(gs[1, i+1])
    df[feat].dropna().hist(bins=40, color=ORANGE, ax=ax, edgecolor="white")
    ax.set_title(feat, fontweight="bold", fontsize=9)
    ax.set_ylabel("Frequency")

# Ordinal features
for i, feat in enumerate(ORDINAL_FEATS[:3]):
    ax = fig.add_subplot(gs[2, i])
    df[feat].value_counts().sort_index().plot(kind="bar", color=PURPLE, ax=ax, edgecolor="white")
    ax.set_title(feat, fontweight="bold", fontsize=9)
    ax.tick_params(axis="x", rotation=0)

fig.suptitle("Exogenous Feature Distributions", fontsize=14, fontweight="bold", y=1.01)
plt.savefig("plots/exog_01_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Univariate Correlation with Target
> Pearson r for numeric/ordinal features, point-biserial r for binary features.
> Shows raw linear relationship strength before any modelling.

In [ ]:
from scipy.stats import pointbiserialr, spearmanr

corr_records = []
for target in TARGETS:
    for feat in ALL_EXOG:
        col = df[feat].fillna(df[feat].median())
        if feat in BINARY_FEATS:
            r, p = pointbiserialr(col, df[target])
            method = "Point-Biserial"
        else:
            r, p = spearmanr(col, df[target])
            method = "Spearman"
        corr_records.append({
            "Feature": feat, "Target": target,
            "Correlation": round(r, 4), "p-value": round(p, 6),
            "Method": method,
            "Significant": "✅" if p < 0.05 else "❌"
        })

corr_df = pd.DataFrame(corr_records)

for target in TARGETS:
    sub = corr_df[corr_df["Target"]==target].sort_values("Correlation", key=abs, ascending=False)
    print(f"\n{'='*65}\n  Correlations with {target}\n{'='*65}")
    print(sub[["Feature","Method","Correlation","p-value","Significant"]].to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_map = {True: GREEN, False: RED}
for ax, target in zip(axes, TARGETS):
    sub = corr_df[corr_df["Target"]==target].sort_values("Correlation")
    colors_ = [GREEN if abs(c)>0.1 and p<0.05 else (ORANGE if p<0.05 else "lightgray")
               for c, p in zip(sub["Correlation"], sub["p-value"])]
    bars = ax.barh(sub["Feature"], sub["Correlation"], color=colors_, edgecolor="white")
    ax.axvline(0, color="black", lw=0.8)
    ax.axvline( 0.1, color="gray", lw=0.8, linestyle="--", alpha=0.5)
    ax.axvline(-0.1, color="gray", lw=0.8, linestyle="--", alpha=0.5)
    for bar, val in zip(bars, sub["Correlation"]):
        xpos = bar.get_width() + 0.002 if val >= 0 else bar.get_width() - 0.002
        ax.text(xpos, bar.get_y()+bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=8,
                ha="left" if val >= 0 else "right")
    ax.set_title(f"Correlation with {target}\n(Green=strong+sig | Orange=sig | Gray=not sig)",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Correlation Coefficient")

plt.tight_layout()
plt.savefig("plots/exog_02_correlation.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Statistical Significance Tests
> ANOVA F-test for binary features (do the two groups have significantly different means?)
> Kruskal-Wallis for ordinal features (non-parametric alternative).

In [ ]:
from scipy.stats import f_oneway, kruskal, mannwhitneyu

print("\n── ANOVA / Mann-Whitney U tests — Binary Features ──")
print(f"{'Feature':<25} {'Target':<22} {'Test':<15} {'Statistic':>10} {'p-value':>12} {'Sig':>5}")
print("-"*85)

sig_records = []
for target in TARGETS:
    for feat in BINARY_FEATS:
        group0 = df[df[feat]==0][target].dropna()
        group1 = df[df[feat]==1][target].dropna()
        stat, p = mannwhitneyu(group0, group1, alternative="two-sided")
        effect  = abs(group1.mean() - group0.mean()) / df[target].std()  # Cohen's d approx
        sig = "✅" if p < 0.05 else "❌"
        print(f"  {feat:<23} {target:<22} Mann-Whitney  {stat:>10.1f} {p:>12.2e} {sig:>5}")
        sig_records.append({"Feature":feat,"Target":target,
                             "Mean(0)":round(group0.mean(),2),
                             "Mean(1)":round(group1.mean(),2),
                             "Delta":round(group1.mean()-group0.mean(),2),
                             "Effect_Size":round(effect,3),
                             "p-value":round(p,6),
                             "Significant": sig})

print("\n\n── Effect Size Summary (Delta = mean[Yes] - mean[No]) ──")
sig_df = pd.DataFrame(sig_records)
print(sig_df[["Feature","Target","Mean(0)","Mean(1)","Delta","Effect_Size","Significant"]].to_string(index=False))


## 5. Volume/Value by Each Binary Feature
> Visual confirmation of the statistical tests above.

In [ ]:
fig, axes = plt.subplots(len(BINARY_FEATS), 2, figsize=(14, 4*len(BINARY_FEATS)))

for row, feat in enumerate(BINARY_FEATS):
    for col, target in enumerate(TARGETS):
        ax = axes[row][col]
        g0 = df[df[feat]==0][target]
        g1 = df[df[feat]==1][target]
        bp = ax.boxplot([g0, g1], patch_artist=True, notch=True,
                        boxprops=dict(facecolor=BLUE if feat!="Is_Weekend" else ORANGE, alpha=0.6),
                        medianprops=dict(color="black", lw=2),
                        flierprops=dict(marker=".", markersize=2, alpha=0.3),
                        widths=0.4)
        bp["boxes"][1].set_facecolor(RED)

        ax.set_xticklabels([f"No\n(n={len(g0):,})", f"Yes\n(n={len(g1):,})"])
        ax.set_title(f"{feat}  →  {target}", fontweight="bold", fontsize=9)
        ax.set_ylabel(target, fontsize=8)

        # Annotate mean difference
        delta = g1.mean() - g0.mean()
        pct   = delta / g0.mean() * 100
        ax.text(1.5, ax.get_ylim()[1]*0.95,
                f"Δ = {delta:+.1f}\n({pct:+.1f}%)",
                ha="center", fontsize=8, color=GREEN if delta > 0 else RED,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

plt.suptitle("Distribution of Targets by Binary Exogenous Features",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plots/exog_03_boxplots_binary.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Day-of-Week Deep Dive
> The most important categorical exogenous feature — quantify the exact effect of each day.

In [ ]:
day_order  = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
day_stats  = (df.groupby("Day_Name")[TARGETS]
                .agg(["mean","std","median"])
                .round(2))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, target in enumerate(TARGETS):
    means = df.groupby("Day_Name")[target].mean().reindex(day_order)
    stds  = df.groupby("Day_Name")[target].std().reindex(day_order)
    overall_mean = df[target].mean()
    pct_diff = ((means - overall_mean) / overall_mean * 100).round(1)
    colors_  = [RED if p < -5 else (GREEN if p > 5 else BLUE) for p in pct_diff]

    # Bar chart with error bars
    axes[0][col].bar(day_order, means, yerr=stds, color=colors_, edgecolor="white",
                     capsize=4, alpha=0.85)
    axes[0][col].axhline(overall_mean, color="black", lw=1.5, linestyle="--",
                          label=f"Overall mean ({overall_mean:.1f})")
    for i, (d, p) in enumerate(zip(day_order, pct_diff)):
        axes[0][col].text(i, means[d] + stds[d] + overall_mean*0.005,
                           f"{p:+.1f}%", ha="center", fontsize=8,
                           color=RED if p < -5 else (GREEN if p > 5 else "gray"))
    axes[0][col].set_title(f"Avg {target} by Day of Week", fontweight="bold")
    axes[0][col].tick_params(axis="x", rotation=30)
    axes[0][col].legend(fontsize=8)

    # Violin plot
    df_day = df.copy()
    df_day["Day_Name"] = pd.Categorical(df_day["Day_Name"], categories=day_order, ordered=True)
    sns.violinplot(data=df_day, x="Day_Name", y=target, ax=axes[1][col],
                   palette=["#D62728" if d in ["Saturday","Sunday"] else "#1A6FBF" for d in day_order],
                   inner="box", cut=0)
    axes[1][col].set_title(f"{target} Distribution by Day (Violin)", fontweight="bold")
    axes[1][col].tick_params(axis="x", rotation=30)
    axes[1][col].set_xlabel("")

plt.suptitle("Day-of-Week Effect on UPI Transactions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/exog_04_dayofweek.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDay-of-Week % deviation from overall mean:")
for target in TARGETS:
    means = df.groupby("Day_Name")[target].mean().reindex(day_order)
    overall = df[target].mean()
    print(f"\n  {target}")
    for d, m in means.items():
        print(f"    {d:<12} {m:>8.2f}  ({(m-overall)/overall*100:+.1f}%)")


## 7. Month & Quarter Seasonality
> Quantify intra-year patterns — which months consistently over/underperform?

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, target in enumerate(TARGETS):
    # Monthly means with CI (across years)
    monthly  = df.groupby(["YearStr","Month_Num"])[target].mean().reset_index()
    m_mean   = monthly.groupby("Month_Num")[target].mean()
    m_std    = monthly.groupby("Month_Num")[target].std()
    overall  = df[target].mean()
    pct      = ((m_mean - overall) / overall * 100)
    colors_  = [GREEN if p > 3 else (RED if p < -3 else BLUE) for p in pct]

    axes[0][col].bar(month_names, m_mean, yerr=m_std, color=colors_,
                     edgecolor="white", capsize=3, alpha=0.85)
    axes[0][col].axhline(overall, color="black", lw=1.5, linestyle="--",
                          label=f"Overall mean ({overall:.1f})")
    for i, p in enumerate(pct):
        axes[0][col].text(i, m_mean.iloc[i]+m_std.iloc[i]+overall*0.005,
                           f"{p:+.1f}%", ha="center", fontsize=8,
                           color=GREEN if p>3 else (RED if p<-3 else "gray"))
    axes[0][col].set_title(f"Monthly Avg {target} (±1 std across years)", fontweight="bold")
    axes[0][col].legend(fontsize=8)

    # Quarterly
    q_mean   = df.groupby("Quarter")[target].mean()
    q_std    = df.groupby("Quarter")[target].std()
    q_labels = ["Q1
(Jan-Mar)","Q2
(Apr-Jun)","Q3
(Jul-Sep)","Q4
(Oct-Dec)"]
    axes[1][col].bar(q_labels, q_mean, yerr=q_std, color=[BLUE,ORANGE,GREEN,RED],
                     edgecolor="white", capsize=4, alpha=0.85, width=0.5)
    axes[1][col].axhline(overall, color="black", lw=1.5, linestyle="--")
    for i, (m, s) in enumerate(zip(q_mean, q_std)):
        axes[1][col].text(i, m+s+overall*0.005,
                           f"{(m-overall)/overall*100:+.1f}%",
                           ha="center", fontsize=9,
                           color=GREEN if m>overall else RED)
    axes[1][col].set_title(f"Quarterly Avg {target}", fontweight="bold")

plt.suptitle("Monthly & Quarterly Seasonality", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/exog_05_month_quarter.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Holiday Cluster & Proximity Effects
> How does transaction volume change as you get closer to / further from a holiday?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, target in enumerate(TARGETS):
    # Holiday cluster effect
    cluster_mean = df.groupby("Holiday_Cluster_7D")[target].agg(["mean","std","count"])
    overall = df[target].mean()

    axes[0][col].bar(cluster_mean.index.astype(str),
                     cluster_mean["mean"], yerr=cluster_mean["std"],
                     color=[RED if m<overall else GREEN for m in cluster_mean["mean"]],
                     edgecolor="white", capsize=3, alpha=0.85)
    axes[0][col].axhline(overall, color="black", lw=1.5, linestyle="--",
                          label=f"Overall mean ({overall:.1f})")
    axes[0][col].set_xlabel("Holiday_Cluster_7D (holidays within ±3 days)")
    axes[0][col].set_title(f"Holiday Cluster Effect — {target}", fontweight="bold")
    axes[0][col].legend(fontsize=8)
    for i, (idx, row_) in enumerate(cluster_mean.iterrows()):
        axes[0][col].text(i, row_["mean"]+row_["std"]+overall*0.005,
                           f"n={int(row_['count'])}", ha="center", fontsize=7, color="gray")

    # Days to next holiday (binned)
    df["DaysToNext_bin"] = pd.cut(df["Days_To_Next_Holiday"].fillna(30),
                                   bins=[0,1,3,7,14,30,999],
                                   labels=["0-1","2-3","4-7","8-14","15-30","30+"])
    dtn_mean = df.groupby("DaysToNext_bin", observed=True)[target].mean()
    axes[1][col].bar(dtn_mean.index.astype(str), dtn_mean.values,
                     color=PURPLE, edgecolor="white", alpha=0.85)
    axes[1][col].axhline(overall, color="black", lw=1.5, linestyle="--")
    axes[1][col].set_xlabel("Days To Next Holiday")
    axes[1][col].set_title(f"Proximity to Next Holiday — {target}", fontweight="bold")
    for i, (idx, val) in enumerate(dtn_mean.items()):
        axes[1][col].text(i, val + overall*0.005,
                           f"{(val-overall)/overall*100:+.1f}%",
                           ha="center", fontsize=8,
                           color=GREEN if val>overall else RED)

plt.suptitle("Holiday Cluster & Proximity Effects", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/exog_06_holiday_cluster_proximity.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Feature × Feature Interaction Heatmap
> Do features co-occur? High co-occurrence means multicollinearity — avoid using both in the same model.

In [ ]:
feat_matrix = df[ALL_EXOG + TARGETS].copy()
feat_matrix = feat_matrix.fillna(feat_matrix.median())

# Spearman correlation (handles ordinal + binary)
from scipy.stats import spearmanr
corr_matrix, _ = spearmanr(feat_matrix)
corr_df_full = pd.DataFrame(corr_matrix,
                              index=feat_matrix.columns,
                              columns=feat_matrix.columns)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Full heatmap
mask = np.triu(np.ones_like(corr_df_full, dtype=bool))
sns.heatmap(corr_df_full, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=axes[0], square=True, linewidths=0.4,
            cbar_kws={"shrink":0.7},
            annot_kws={"size":7})
axes[0].set_title("Spearman Correlation — All Features + Targets",
                   fontweight="bold", fontsize=10)

# Feature-only collinearity check
feat_corr = corr_df_full.loc[ALL_EXOG, ALL_EXOG]
sns.heatmap(feat_corr, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, ax=axes[1], square=True, linewidths=0.4,
            cbar_kws={"shrink":0.7},
            annot_kws={"size":8})
axes[1].set_title("Feature-Feature Collinearity\n(|r|>0.7 = multicollinearity risk)",
                   fontweight="bold", fontsize=10)

# Flag high collinearity pairs
high_pairs = []
for i in range(len(ALL_EXOG)):
    for j in range(i+1, len(ALL_EXOG)):
        r = feat_corr.iloc[i,j]
        if abs(r) > 0.5:
            high_pairs.append((ALL_EXOG[i], ALL_EXOG[j], round(r,3)))

plt.tight_layout()
plt.savefig("plots/exog_07_feature_interaction.png", dpi=150, bbox_inches="tight")
plt.show()

if high_pairs:
    print("\n⚠️  High collinearity pairs (|r| > 0.5):")
    for f1, f2, r in sorted(high_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {f1:<30} ↔  {f2:<30}  r={r:+.3f}")
    print("\n  → Avoid using both features from a high-collinearity pair in SARIMAX.")
else:
    print("✅ No high collinearity pairs found (all |r| ≤ 0.5).")


## 10. Mutual Information & Random Forest Importance
> Mutual Information captures non-linear relationships that Pearson/Spearman miss.
> Random Forest importance ranks features by their contribution to predicting the target.

In [ ]:
feat_df = df[ALL_EXOG + TARGETS].fillna(df[ALL_EXOG + TARGETS].median())

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for col, target in enumerate(TARGETS):
    X = feat_df[ALL_EXOG].values
    y = feat_df[target].values

    # Mutual Information
    mi_scores = mutual_info_regression(X, y, random_state=42)
    mi_df = pd.DataFrame({"Feature": ALL_EXOG, "MI Score": mi_scores}).sort_values("MI Score")

    colors_ = [GREEN if s > mi_df["MI Score"].median() else BLUE for s in mi_df["MI Score"]]
    axes[0][col].barh(mi_df["Feature"], mi_df["MI Score"],
                      color=colors_, edgecolor="white")
    axes[0][col].set_title(f"Mutual Information — {target}", fontweight="bold")
    axes[0][col].set_xlabel("MI Score (higher = more informative)")
    for i, (_, row_) in enumerate(mi_df.iterrows()):
        axes[0][col].text(row_["MI Score"]+mi_df["MI Score"].max()*0.01,
                           i, f"{row_['MI Score']:.4f}",
                           va="center", fontsize=8)

    # Random Forest importance
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1, max_depth=8)
    rf.fit(X, y)
    rf_df = pd.DataFrame({
        "Feature": ALL_EXOG,
        "Importance": rf.feature_importances_
    }).sort_values("Importance")

    colors_rf = [RED if s > rf_df["Importance"].quantile(0.75) else
                 (ORANGE if s > rf_df["Importance"].median() else "lightgray")
                 for s in rf_df["Importance"]]
    axes[1][col].barh(rf_df["Feature"], rf_df["Importance"],
                      color=colors_rf, edgecolor="white")
    axes[1][col].set_title(f"Random Forest Importance — {target}", fontweight="bold")
    axes[1][col].set_xlabel("Feature Importance (impurity-based)")
    for i, (_, row_) in enumerate(rf_df.iterrows()):
        axes[1][col].text(row_["Importance"]+rf_df["Importance"].max()*0.005,
                           i, f"{row_['Importance']:.4f}",
                           va="center", fontsize=8)

plt.suptitle("Mutual Information & Random Forest Feature Importance",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/exog_08_mi_rf_importance.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. SARIMAX: With vs Without Exogenous Features
> The key model impact test — does adding exogenous features actually improve forecast accuracy?

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# Use a fixed simple order for fair comparison
ORDER         = (1, 1, 1)
SEASONAL_ORDER = (1, 0, 1, 7)
EXOG_COLS     = ["Is_Weekend","Is_Festival","Is_Holiday_Adjacent",
                 "Is_Long_Weekend","Holiday_Cluster_7D"]

comparison_results = []

for TARGET in TARGETS:
    train_y    = train.set_index("Date")[TARGET]
    test_y     = test.set_index("Date")[TARGET]
    train_exog = train[EXOG_COLS].values
    test_exog  = test[EXOG_COLS].values

    # ── Without exog ─────────────────────────────────────────────
    print(f"Fitting SARIMA (no exog) — {TARGET}...")
    m_no = SARIMAX(train_y, order=ORDER, seasonal_order=SEASONAL_ORDER,
                   enforce_stationarity=False).fit(disp=False)
    fc_no  = m_no.forecast(len(test_y)).values
    mae_no = np.mean(np.abs(test_y.values - fc_no))
    mape_no= np.mean(np.abs((test_y.values - fc_no) / test_y.values)) * 100

    # ── With exog ─────────────────────────────────────────────────
    print(f"Fitting SARIMAX (with exog) — {TARGET}...")
    m_ex = SARIMAX(train_y, exog=train_exog, order=ORDER, seasonal_order=SEASONAL_ORDER,
                   enforce_stationarity=False).fit(disp=False)
    fc_ex  = m_ex.forecast(len(test_y), exog=test_exog).values
    mae_ex = np.mean(np.abs(test_y.values - fc_ex))
    mape_ex= np.mean(np.abs((test_y.values - fc_ex) / test_y.values)) * 100

    improvement_mae  = (mae_no - mae_ex)  / mae_no  * 100
    improvement_mape = (mape_no - mape_ex)/ mape_no * 100

    comparison_results.append({
        "Target": TARGET,
        "SARIMA MAE":   round(mae_no, 2),  "SARIMA MAPE":  round(mape_no, 2),
        "SARIMAX MAE":  round(mae_ex, 2),  "SARIMAX MAPE": round(mape_ex, 2),
        "MAE Improvement%":  round(improvement_mae, 2),
        "MAPE Improvement%": round(improvement_mape, 2),
    })

    print(f"\n── {TARGET} ──")
    print(f"  SARIMA  (no exog): MAE={mae_no:.2f}  MAPE={mape_no:.2f}%")
    print(f"  SARIMAX (w/ exog): MAE={mae_ex:.2f}  MAPE={mape_ex:.2f}%")
    sign = "✅ IMPROVED" if improvement_mape > 0 else "❌ WORSE"
    print(f"  {sign} — MAPE Δ = {improvement_mape:+.2f}%  |  MAE Δ = {improvement_mae:+.2f}%")

    # Plot comparison
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    axes[0].plot(test_y.index, test_y.values,  color="black",  lw=2,   label="Actual")
    axes[0].plot(test_y.index, fc_no,           color=ORANGE,   lw=1.5, linestyle="--",  label=f"SARIMA (no exog)  MAPE={mape_no:.2f}%")
    axes[0].plot(test_y.index, fc_ex,           color=GREEN,    lw=1.5, linestyle="-.",  label=f"SARIMAX (w/ exog) MAPE={mape_ex:.2f}%")
    axes[0].set_title(f"SARIMA vs SARIMAX — {TARGET}", fontsize=12, fontweight="bold")
    axes[0].set_ylabel(TARGET); axes[0].legend(fontsize=9)

    axes[1].plot(test_y.index, test_y.values - fc_no, color=ORANGE, lw=1, label="Residual — no exog", alpha=0.8)
    axes[1].plot(test_y.index, test_y.values - fc_ex, color=GREEN,  lw=1, label="Residual — w/ exog", alpha=0.8)
    axes[1].axhline(0, color="black", lw=0.8, linestyle="--")
    axes[1].set_title("Residuals Comparison", fontweight="bold")
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f"plots/exog_09_sarima_vs_sarimax_{TARGET[:3].lower()}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

print("\n── Summary Table ──")
print(pd.DataFrame(comparison_results).to_string(index=False))


## 12. SARIMAX Coefficient Analysis
> Which exogenous features are statistically significant in the model? Insignificant ones should be dropped.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

EXOG_COLS = ["Is_Weekend","Is_Festival","Is_Holiday_Adjacent",
             "Is_Long_Weekend","Holiday_Cluster_7D"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for col, TARGET in enumerate(TARGETS):
    train_y    = train.set_index("Date")[TARGET]
    train_exog = train[EXOG_COLS].values

    model  = SARIMAX(train_y, exog=train_exog, order=(1,1,1),
                     seasonal_order=(1,0,1,7),
                     enforce_stationarity=False).fit(disp=False)

    coef_df = pd.DataFrame({
        "Feature":   EXOG_COLS,
        "Coef":      [model.params.get(c, np.nan) for c in EXOG_COLS],
        "Std_Err":   [model.bse.get(c, np.nan)    for c in EXOG_COLS],
        "p_value":   [model.pvalues.get(c, np.nan) for c in EXOG_COLS],
    }).dropna()
    coef_df["Significant"] = coef_df["p_value"] < 0.05
    coef_df["CI_lo"] = coef_df["Coef"] - 1.96 * coef_df["Std_Err"]
    coef_df["CI_hi"] = coef_df["Coef"] + 1.96 * coef_df["Std_Err"]
    coef_df = coef_df.sort_values("Coef")

    colors_  = [GREEN if (s and c>0) else (RED if (s and c<0) else "lightgray")
                for s, c in zip(coef_df["Significant"], coef_df["Coef"])]
    ax = axes[col]
    bars = ax.barh(coef_df["Feature"], coef_df["Coef"], color=colors_, edgecolor="white")
    ax.errorbar(coef_df["Coef"], range(len(coef_df)),
                xerr=1.96*coef_df["Std_Err"],
                fmt="none", color="black", capsize=4, lw=1.2)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"SARIMAX Exog Coefficients — {TARGET}\n"
                 f"(Green=sig+pos | Red=sig+neg | Gray=not sig)",
                 fontweight="bold", fontsize=9)
    ax.set_xlabel("Coefficient (±95% CI)")

    for i, row_ in enumerate(coef_df.itertuples()):
        sig_marker = "✅" if row_.Significant else "❌"
        ax.text(coef_df["Coef"].max()*0.02 + coef_df["CI_hi"].max(),
                i, f"  p={row_.p_value:.3f} {sig_marker}",
                va="center", fontsize=8)

    print(f"\n{TARGET} — SARIMAX Coefficient Summary:")
    print(coef_df[["Feature","Coef","p_value","Significant"]].to_string(index=False))

plt.tight_layout()
plt.savefig("plots/exog_10_sarimax_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()


## 13. Prophet Regressor Importance
> Prophet doesn't give direct p-values for regressors, but we can use permutation importance to measure contribution.

In [ ]:
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False
    print("Prophet not installed. Run: pip install prophet")

if PROPHET_AVAILABLE:
    EXTRA_REGS = ["Is_Weekend","Is_Festival","Is_Long_Weekend","Holiday_Cluster_7D"]

    for TARGET in TARGETS:
        train_df = train[["Date", TARGET] + EXTRA_REGS].rename(
            columns={"Date":"ds", TARGET:"y"})
        test_df  = test[["Date",  TARGET] + EXTRA_REGS].rename(
            columns={"Date":"ds", TARGET:"y"})
        for col_ in EXTRA_REGS:
            train_df[col_] = train_df[col_].astype(float)
            test_df[col_]  = test_df[col_].astype(float)

        model = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                        daily_seasonality=False, seasonality_mode="multiplicative",
                        changepoint_prior_scale=0.05)
        for col_ in EXTRA_REGS:
            model.add_regressor(col_)
        model.fit(train_df)

        # Permutation importance: shuffle one feature at a time, measure MAPE increase
        base_fc   = model.predict(test_df)["yhat"].values
        actual    = test_df["y"].values
        base_mape = np.mean(np.abs((actual - base_fc) / actual)) * 100

        perm_results = []
        for feat in EXTRA_REGS:
            test_perm = test_df.copy()
            test_perm[feat] = np.random.permutation(test_perm[feat].values)
            perm_fc   = model.predict(test_perm)["yhat"].values
            perm_mape = np.mean(np.abs((actual - perm_fc) / actual)) * 100
            perm_results.append({
                "Feature": feat,
                "MAPE Increase": round(perm_mape - base_mape, 4),
                "Importance": round(perm_mape - base_mape, 4)
            })

        perm_df = pd.DataFrame(perm_results).sort_values("MAPE Increase", ascending=True)

        fig, ax = plt.subplots(figsize=(9, 4))
        colors_ = [GREEN if v > 0 else RED for v in perm_df["MAPE Increase"]]
        ax.barh(perm_df["Feature"], perm_df["MAPE Increase"], color=colors_, edgecolor="white")
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel("MAPE increase when feature is shuffled\n(higher = more important)")
        ax.set_title(f"Prophet Permutation Importance — {TARGET}", fontweight="bold")
        for i, row_ in enumerate(perm_df.itertuples()):
            ax.text(row_.MAPE_Increase + 0.001, i,
                    f"{row_.MAPE_Increase:+.3f}%", va="center", fontsize=9)
        plt.tight_layout()
        plt.savefig(f"plots/exog_11_prophet_perm_importance_{TARGET[:3].lower()}.png",
                    dpi=150, bbox_inches="tight")
        plt.show()

        print(f"\nProphet Permutation Importance — {TARGET}:")
        print(f"  Baseline MAPE: {base_mape:.3f}%")
        print(perm_df.to_string(index=False))


## 14. Final Feature Recommendation Table
> Synthesises all analysis above into a clear recommendation for each model.

In [ ]:
recommendations = pd.DataFrame([
    {
        "Feature":             "Is_Weekend",
        "Correlation":         "Strong negative (Sunday dips)",
        "Stat. Significant":   "✅ Yes",
        "MI Rank":             "High",
        "Use in SARIMAX":      "✅ Yes",
        "Use in Prophet":      "✅ Yes (add_regressor)",
        "Use in LSTM":         "✅ Yes (multivariate)",
        "Notes":               "Most impactful binary feature",
    },
    {
        "Feature":             "Is_Festival",
        "Correlation":         "Moderate negative (holiday suppression)",
        "Stat. Significant":   "✅ Yes",
        "MI Rank":             "Medium",
        "Use in SARIMAX":      "✅ Yes",
        "Use in Prophet":      "✅ Via holidays DataFrame",
        "Use in LSTM":         "✅ Yes",
        "Notes":               "Pass as holidays to Prophet, not regressor",
    },
    {
        "Feature":             "Is_Holiday_Adjacent",
        "Correlation":         "Weak",
        "Stat. Significant":   "Conditional",
        "MI Rank":             "Low–Medium",
        "Use in SARIMAX":      "✅ Yes",
        "Use in Prophet":      "✅ Via lower/upper_window",
        "Use in LSTM":         "✅ Yes",
        "Notes":               "Prophet handles this natively with window params",
    },
    {
        "Feature":             "Is_Long_Weekend",
        "Correlation":         "Moderate negative",
        "Stat. Significant":   "✅ Yes",
        "MI Rank":             "Medium",
        "Use in SARIMAX":      "✅ Yes",
        "Use in Prophet":      "✅ Yes (add_regressor)",
        "Use in LSTM":         "✅ Yes",
        "Notes":               "Correlated with Is_Weekend — watch multicollinearity",
    },
    {
        "Feature":             "Holiday_Cluster_7D",
        "Correlation":         "Moderate negative",
        "Stat. Significant":   "✅ Yes",
        "MI Rank":             "Medium",
        "Use in SARIMAX":      "✅ Yes",
        "Use in Prophet":      "✅ Yes (add_regressor)",
        "Use in LSTM":         "✅ Yes",
        "Notes":               "Captures cumulative holiday density",
    },
    {
        "Feature":             "Days_To_Next_Holiday",
        "Correlation":         "Weak",
        "Stat. Significant":   "❌ Often not",
        "MI Rank":             "Low",
        "Use in SARIMAX":      "❌ Drop",
        "Use in Prophet":      "❌ Redundant with holidays",
        "Use in LSTM":         "⚠️  Optional",
        "Notes":               "Many NaN values; redundant with Is_Holiday_Adjacent",
    },
    {
        "Feature":             "Days_Since_Prev_Holiday",
        "Correlation":         "Weak",
        "Stat. Significant":   "❌ Often not",
        "MI Rank":             "Low",
        "Use in SARIMAX":      "❌ Drop",
        "Use in Prophet":      "❌ Redundant",
        "Use in LSTM":         "⚠️  Optional",
        "Notes":               "Many NaN values at start of series",
    },
    {
        "Feature":             "Day_Number",
        "Correlation":         "Strong (captures weekend cycle)",
        "Stat. Significant":   "✅ Yes",
        "MI Rank":             "High",
        "Use in SARIMAX":      "⚠️  Redundant with s=7 seasonal",
        "Use in Prophet":      "⚠️  Redundant with weekly_seasonality",
        "Use in LSTM":         "✅ Yes — helps learn weekly cycle",
        "Notes":               "Avoid in SARIMA/Prophet; useful for LSTM",
    },
    {
        "Feature":             "Month_Num",
        "Correlation":         "Moderate (yearly seasonality)",
        "Stat. Significant":   "✅ Yes",
        "MI Rank":             "Medium–High",
        "Use in SARIMAX":      "⚠️  Redundant with s=12 or yearly seasonal",
        "Use in Prophet":      "⚠️  Redundant with yearly_seasonality",
        "Use in LSTM":         "✅ Yes",
        "Notes":               "Useful for LSTM multivariate input",
    },
])

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
print("\n── Final Exogenous Feature Recommendations ──")
print(recommendations.to_string(index=False))

# Save as CSV
recommendations.to_csv("models/exog_feature_recommendations.csv", index=False)
print("\n✅ Saved to models/exog_feature_recommendations.csv")
print("\n── Recommended SARIMAX feature set ──")
print("  [Is_Weekend, Is_Festival, Is_Holiday_Adjacent, Is_Long_Weekend, Holiday_Cluster_7D]")
print("\n── Recommended LSTM feature set ──")
print("  [Is_Weekend, Is_Festival, Is_Holiday_Adjacent, Is_Long_Weekend,")
print("   Holiday_Cluster_7D, Day_Number, Month_Num]")
